# Repasando FastAPI paso a paso

En este Notebook repasaremos progresivamente los conceptos principales de FastAPI.

Cada sección retomará un concepto, lo pondrá en contexto e incorporará el código necesario para probarlo.


# 1. Instalación de FastAPI

### ¿Qué estamos instalando?

`FastAPI` es el framework que utilizaremos para construir nuestra API.

Al instalar:

```python
%pip install "fastapi[standard]"

# 2. Creación de la aplicación

Primero importamos la clase `FastAPI`.

Después creamos una instancia y la guardamos en la variable `app`. Este objeto representará nuestra aplicación.

In [1]:
from fastapi import FastAPI

app = FastAPI()

### Comprobación

Utilizamos 'type()' para comprobar que clase de objeto se guardo en la vaiable 'app'. 

In [2]:
type(app)

fastapi.applications.FastAPI

# 3. Ruta de prueba: Hello World

Esta ruta responde cuando alguien solicita la direccion principal de la API: '/'.

In [3]:
@app.get('/')
async def read_root():
    return{"message": "Hello, World"}

### Nota: rutas repetidas

FastAPI evalúa las rutas en el orden en que fueron registradas. Cuando recibe GET/, encuentra la primera coincidencia (read_root) y la ejecuta. Como ya encontró una ruta válida, no sigue buscando otra. Esto permite que rutas especificas se declaren antes que rutas variables, por ejemplo, /user/me antes de /user/{user_id}.

In [4]:
# Aplicamos para la misma ruta otra funcion a ver que pasa.
""" @app.get('/')
async def read_root2():
    return{"message": "Hello, World2"} """

' @app.get(\'/\')\nasync def read_root2():\n    return{"message": "Hello, World2"} '

## 3. Ejecucion

Si dos funciones usan la misma combinación de método y ruta, por ejemplo `GET /`, FastAPI ejecuta la primera que fue registrada.
___
![Resultado de Hello World](Images/03%20Hello%20world.png)
___
Sin embargo, la documentación automática (`/docs`) solo puede mostrar una definición para `GET /`, por lo que termina mostrando la última.

Esto genera una inconsistencia: la documentación puede describir una ruta distinta de la que realmente se ejecuta. Por eso cada combinación de método HTTP y ruta debe ser única.
___
/
___
![Resultado de docs](Images/03%20Hello%20world%20docs.png)

## 3. Explicación didáctica

### ¿Qué hace esta ruta?

Esta es la ruta más simple de la API.

Cuando un cliente realiza:

`GET /`

FastAPI ejecuta la función asociada y devuelve su resultado.

### ¿Dónde se suele usar?

La ruta `/` suele utilizarse como página inicial o como una comprobación sencilla de que la API está funcionando.

# 4. Una segunda ruta: `/saludo`
Una ruta identifica una direccion de la API. `@app.get("/saludo")` registra la funcion siguiente para responder solicitudes `GET` a esa direccion.
El nombre `read_greeting` es elegido por nosotros; FastAPI usa la combinacion `GET /saludo` para encontrarla.

In [5]:
@app.get("/saludo")
async def read_greeting():
    return{"message": "Usamos una ruta"}

## 4. Ejecucion

![Hola Ramiro](Images\04%20Segunda%20ruta.png)
___
/saludo
___
![Hola Ramiro docs](Images\04%20Segunda%20ruta%20docs.png)


## 4. Explicación didáctica

### ¿Qué estamos agregando?

Una API puede tener muchas rutas diferentes.

```python
@app.get("/saludo")
```
indica que esta función debe ejecutarse cuando llegue:

`GET /saludo`

El decorador conecta un método HTTP y una ruta con una función de Python.

Ejemplos de rutas
- `saludo` → devolver un saludo.
- `tasks` → consultar tareas.
- `users` → consultar usuarios.
- `products` → consultar productos.

# 5. Parametros de ruta

`{task_id}` representa una parte variable de la URL. En `/tasks/5`, FastAPI recibe 5 y lo entrega a `task_id`.
La anotacion `: init` egige un numero entero; si se escribe `/tasks/hola`, FastAPI devuelve un Error de validacion.

In [6]:
@app.get("/tasks/{task_id}")
async def read_task(task_id: int):
    return{"task_id": task_id}

## 5. Ejecucion

![Tasks](Images\05%20Parametros%20ruta.png)
___
/tasks/{task_id}
___
![Tasks docs](Images\05%20Parametros%20ruta%20docs.png)

## 5. Explicación didáctica

### ¿Qué son los parámetros de ruta?

Los parámetros de ruta permiten identificar un recurso concreto mediante un valor incluido directamente en la URL.

Por ejemplo:

`GET /tasks/5`

significa: "quiero la tarea número 5".

En `/tasks/{task_id}`, FastAPI coloca automáticamente el `5` dentro de la variable `task_id`.

### Usos típicos

Se utilizan habitualmente para identificar:

- `/tasks/5` → una tarea.
- `/users/20` → un usuario.
- `/products/150` → un producto.
- `/orders/30` → un pedido.

# 6. Parámetros de consulta

Los parámetros de consulta son opcionales y modifican una consulta, por ejemplo `/tasks?completed=true&limit=5`; `?` inicia esa parte de la URL.

`bool | None = None` permite `True`, `False` o ausencia de filtro; así `None` significa “traer todas”.

Usamos `{task_id}` cuando el dato identifica obligatoriamente un recurso, como `/tasks/5`; usamos `?` para filtros, orden, paginación o límites que no cambian cuál es el recurso principal.

In [7]:
@app.get("/tasks")
async def read_tasks(completed: bool | None = None, limit: int = 10):
    return{"completed": completed, "limit": limit}

## 6. Ejecucion

![Parametros consulta](Images\06%20Parametros%20consulta.png)
___
/tasks
___
![Parametros consulta docs](Images\06%20Parametros%20consulta%20docs.png)

## 6. Explicación didáctica

### ¿Qué son los parámetros de consulta?

Los parámetros de consulta permiten agregar **filtros u opciones** a una solicitud sin cambiar el recurso principal.

Por ejemplo:

`GET /tasks?completed=true`

seguimos solicitando `/tasks`, pero pedimos solamente las tareas completadas.

### Usos típicos

- `completed=true` → filtrar por estado.
- `limit=10` → limitar la cantidad de resultados.
- `skip=20` → saltar resultados para paginación.
- `sort=date` → ordenar por un campo.
- `search=fastapi` → buscar por texto.
- `min_age=18&max_age=30` → filtrar por un rango numérico.
- `from_date=2026-09-01&to_date=2026-09-30` → filtrar por un rango de fechas.

### Parámetro de ruta vs parámetro de consulta

```text
/tasks/5               → quiero específicamente la tarea 5
/tasks?completed=true  → quiero tareas, pero filtradas
```

# 7. Validacion de parametros de consulta

`Query()` agrega reglas especificas para un parametro recibido desde la URL. `ge=1` significa "mayor o igual que 1" y `le=100`, "menor o igual que 100".

`Annotated` une el tipo `ìnt` con esas reglas; si `limit` queda fuera de ese rango. FastAPI rechaza la solicitud automaticamente con un error de validacion.

In [8]:
from typing import Annotated
from fastapi import Query

@app.get("/tasks-filtered")
async def read_filtered_tasks(completed:bool | None = None, limit: Annotated[int, Query(ge=1, le=100)]=10):
    return{"completed": completed, "limit": limit}

### Alcance de `Annotated` y `Query`
`Annotated` permite asociar información adicional a un tipo: aquí indica que `limit` es un `int` y que sus reglas provienen de `Query`.
Además de `ge` y `le`, `Query` puede validar longitudes, patrones, alias, valores obligatorios y descripciones para `/docs`.
El mismo mecanismo se usa más adelante con `Path`, `Header`, `Body` y `Depends`; no los aplicamos aún para incorporar una idea por vez.

## 7. Ejecucion

![Validacion parametros consulta](Images\07%20Validacion%20parametros%20consulta.png)
___
/tasks-filtered
___
![Validacion parametros consulta docs](Images\07%20Validacion%20parametros%20consulta%20docs.png)

## 7. Explicación didáctica

### ¿Para qué sirve `Query()`?

`Query()` permite establecer reglas que deben cumplir los parámetros de consulta.

Por ejemplo:

```python
Query(ge=1, le=100)
```
significa que el valor debe estar entre 1 y 100.

ge → greater or equal → mayor o igual.
le → less or equal → menor o igual.

FastAPI comprueba estas reglas automáticamente antes de ejecutar nuestra función.

Usos típicos

Se puede utilizar para validar:

- valores mínimos y máximos;
- longitud de textos;
- formatos determinados;
- parámetros obligatorios;
- descripciones para `/docs.`

Los casos 8 a 11 siguen directamente el flujo de creación, validación y respuesta.

# 8. Crear datos con `POST` y un modelo.

`POST` se usa para enviar datos nuevos a la API. `BaseModel` define la estructura que esperamos recibir en el cuerpo JSON  y FastAPI la valida automaticamente.

In [9]:
from pydantic import BaseModel

class TaskCreate(BaseModel):
    title: str
    completed: bool = False

@app.post("/tasks-created")
async def create_task(task: TaskCreate):
    return{"message": "Tarea creada", "task": task}

## 8. Ejecucion

### ¿Para qué se usa `POST`?
`POST` se utiliza para crear recursos enviando datos al servidor: por ejemplo, una app web, móvil o frontend envía una tarea nueva como JSON.  
Al abrir `/tasks-created` en el explorador, este realiza una solicitud `GET`; como la ruta solo acepta `POST`, FastAPI responde `{"detail":"Method Not Allowed"}`.  
Para probarla usamos `/docs` → `POST /tasks-created` → **Try it out**, o herramientas como Postman, `curl` o código Python.
___
/tasks-created
___
![Datos con POST](Images\08%20Post%20docs.png)

## 8. Explicación didáctica

### ¿Para qué se usa `POST`?

`POST` se utiliza normalmente cuando el cliente quiere **crear un recurso nuevo**.

El cliente envía los datos dentro del cuerpo de la solicitud, generalmente como JSON.

`BaseModel` define qué estructura deben tener esos datos y FastAPI los valida antes de entregarlos a nuestra función.

### Usos típicos

- crear una tarea;
- registrar un usuario;
- crear un pedido;
- publicar un comentario;
- guardar una medición.

# 9. Validar el cuerpo de una solicitud.
`Field()` agrega reglas a cada dato del modelo. Asi evitamos crear tareas sin titulo o con un titulo demaciado corto.

Si el JSON no cumple estas reglas, FastAPI responde con un error `422` antes de ejecutarla funcion.

In [10]:
from pydantic import BaseModel, Field

class ValidatedTaskCreated(BaseModel):
    title: str = Field(min_length=3, max_length=100)
    completed: bool=False

@app.post("/tasks-validated")
async def create_validated_task(task: ValidatedTaskCreated):
    return{"message": "Tarea validada", "task": task}


## 9. Ejecucion

Insertamos:

```json
{
  "title": "A",
  "completed": false
}
```

Sabemos que `"A"` tiene menos de tres caracteres.
___
/tasks-validated
___
![Datos con POST](Images\09%20Validar%20post%20docs.png)

## 9. Explicación didáctica

### ¿Para qué sirve `Field()`?

`Field()` permite agregar reglas más específicas a los campos de un modelo.

Por ejemplo:

```python
Field(min_length=3, max_length=100)
```
indica que el texto debe tener entre 3 y 100 caracteres.

Si los datos no cumplen las reglas, FastAPI rechaza la solicitud antes de ejecutar nuestra función.

Usos típicos

Permite validar:

- longitud mínima y máxima;
- valores mínimos y máximos;
- determinados formatos;
- restricciones específicas de cada campo.

# 10. Codigos de estado HTTP

Ademas de JSON, una API comunica el resultado mediante un codigo HTTP. `201 Created` indica que el servidor creó un recurso nuevo.

Usamos la constante `status.HTTP_201_CREATED` en lugar del numero `201` para que el codigo sea mas legible.

`200 OK` significa: “la operación salió bien”. Es el valor predeterminado de FastAPI si no indicamos otro.
`201 Created` significa algo más preciso: “la operación salió bien y además se creó un recurso nuevo”.

In [11]:
from fastapi import status

@app.post("/tasks-created-with-status", status_code=status.HTTP_201_CREATED)
async def create_task_with_status(task: ValidatedTaskCreated):
    return {"message": "Tarea creada correctamente", "task": task}

## 10. Ejecucion

___
/tasks-created-with-status
___
![Codigo HTTP](Images\10%20Codigo%20HTTP%20docs.png)

## 10. Explicación didáctica

### ¿Qué son los códigos de estado HTTP?

El código HTTP informa al cliente **qué ocurrió con su solicitud**.

Por ejemplo:

- `200 OK` → la operación salió correctamente.
- `201 Created` → se creó correctamente un recurso.
- `404 Not Found` → el recurso solicitado no existe.
- `422 Unprocessable Entity` → los datos recibidos no cumplen la validación.

### Idea principal

```text
JSON        → qué información devuelve el servidor
Código HTTP → qué ocurrió con la solicitud
```

# 11. Modelo de respuesta

`response_model` define la estructura que la API enviara al cliente. Es util para documentar, validar y evitar devolver datos internos por error.

El modelo de entrada describe lo que recibimos; el de respuesta describe lo que entregamos. 

In [12]:
class TaskResponse(BaseModel):
    id: int
    title: str
    completed: bool

@app.post("/tasks-response-model", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
async def create_task_response(task: ValidatedTaskCreated):
    return{"id": 1, **task.model_dump()}

## 11. Ejecucion

`task` es un objeto creado a partir de `ValidatedTaskCreate`. `task.model_dump()` lo convierte en un diccionario de Python con sus datos.
El operador `**` desempaqueta ese diccionario dentro de otro; por eso `{"id": 1, **task.model_dump()}` combina el `id` generado por el servidor con `title` y `completed` enviados por el cliente.
___
/tasks-response-model
___
![Modelo respuesta](Images\11%20Modelo%20respuesta%20docs.png)

## 11. Explicación didáctica

### ¿Para qué sirve `response_model`?

`response_model` define qué estructura tendrá la información que nuestro servidor devuelve al cliente.

Sirve para validar y controlar los datos de salida.

### Entrada vs salida

Podemos utilizar un modelo para recibir datos y otro diferente para devolverlos.

Por ejemplo:

```text
TaskCreate   → datos que envía el cliente
TaskResponse → datos que devuelve el servidor
```
Esto permite que el servidor agregue información propia, como un id, o evite devolver datos internos que el cliente no debería recibir.

Los casos 12 a 15 trabajan sobre el almacenamiento temporal y la búsqueda por identificador.

# 12. Almacenamiento temporal en memoria
Por ahora guardaremos las tareas en una lista de Python; funciona como una base de datos muy simple mientras el servidor está activo.

Al reiniciar el kernel, la lista se vacía: más adelante la reemplazaremos por una base de datos real. `next_task_id` genera identificadores consecutivos.

In [13]:
tasks_memory: list[TaskResponse]= []
next_task_id = 1

@app.post("/tasks-memory", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
async def create_task_in_memory(task: ValidatedTaskCreated):
    global next_task_id
    saved_task = TaskResponse(id=next_task_id, **task.model_dump())
    tasks_memory.append(saved_task)
    next_task_id += 1
    return saved_task

## 12. Ejecucion

![Almacenamiento temporal comandos](Images\12%20Almacenamiento%20temporal%20command.png)
___
/tasks-memory
___
![Almacenamiento temporal docs](Images\12%20Almacenamiento%20temporal%20docs.png)

## 12. Explicación didáctica

### ¿Qué estamos haciendo?

Aquí utilizamos una lista de Python como almacenamiento temporal.

Nos permite practicar el funcionamiento de una API sin utilizar todavía una base de datos.

### ¿Cuál es la limitación?

Los datos existen solamente mientras el programa está funcionando.

Si reiniciamos el servidor o el kernel, la lista vuelve a quedar vacía.

En una aplicación real, este almacenamiento normalmente sería reemplazado por PostgreSQL, MySQL u otra base de datos.

# 13. Listar tareas
La misma URL puede aceptar distintos metodos: `POST /tasks-memory` crea una tarea y `GET /tasks-memory` devuelve las existentes.

`response_model = list[TaskResponse]` indica que la respuesta será una lista; si todavia no creamos tareas, devuelve `[]`.

In [14]:
@app.get("/tasks-memory", response_model=list[TaskResponse])
async def read_tasks_memory():
    return tasks_memory

## 13. Ejecucion

![Listar tareas](Images\13%20Listar%20tareas.png)
___
/tasks-memory
___
![Listar tareas docs](Images\13%20Listar%20tareas%20docs.png)

## 13. Explicación didáctica

### ¿Qué estamos haciendo con `GET`?

`GET /tasks-memory` devuelve la colección de tareas almacenadas.

La misma ruta también puede utilizar `POST`, pero cada método realiza una operación diferente.

```text
GET  /tasks-memory → consultar tareas
POST /tasks-memory → crear una tarea
```
### Idea principal

La ruta representa el recurso y el método HTTP indica qué queremos hacer con él.

# 14. Buscar una tarea por su identificador
`{task_id}` recibe el numero escrito en la URL, por ejemplo `/tasks-memory/2`. Recorremos la lista hasta encontrar una tarea con ese `id`.

Por ahora, si no existe, devolvemos `null`; en el siguiente paso lo reemplazamos por el error HTTP correcto.

In [15]:
@app.get("/tasks-memory/{task_id}", response_model=TaskResponse | None)
async def read_task_memory(task_id: int):
    for task in tasks_memory:
        if task.id == task_id:
            return task
    return None

## 14. Ejecucion

![Buscar tareas](Images\14%20Buscar%20tareas%20id%203.png)
___
![Buscar tareas](Images\14%20Buscar%20tareas%20id%207.png)
___
/tasks-memory/{task_id}
___
![Buscar tareas docs](Images\14%20Buscar%20tareas%20id%205%20docs.png)

## 14. Explicación didáctica

### Buscar un recurso por ID

Aquí combinamos `GET` con un parámetro de ruta para buscar un recurso específico.

Por ejemplo:

`GET /tasks-memory/2`

significa:

"Buscá y devolveme la tarea cuyo ID es 2".

### Usos típicos

Este patrón es muy común:

- `GET /users/15`
- `GET /products/20`
- `GET /orders/100`
- `GET /tasks/2`

Todos permiten consultar un elemento concreto mediante su identificador.

# 15. Informar recursos inexistentes con `404`
`null` con codigo `200` puede confundir: la solicitud funcionó, pero no encontramos la tarea. `404 Not Found` expresa correctamente esta situacion.

`HTTPExeption` detiene la funcion y envia una respuesta de error con un mensaje para el cliente.

In [16]:
from fastapi import HTTPException

@app.get("/tasks-memory-safe/{task_id}", response_model=TaskResponse)
async def read_task_memory_safe(task_id:int):
    for task in tasks_memory:
        if task.id == task_id:
            return task
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Task not found")

## 15. Ejecucion

![Recursos inexistentes](Images\15%20Recursos%20inexistentes.png)
___
/tasks-memory-safe/{task_id}
___
![Recursos inexistentes docs](Images\15%20Recursos%20inexistentes%20docs.png)

### ¿Por qué usamos `HTTPException`?

Si buscamos una tarea que no existe, no conviene responder `200 OK`, porque ese código indica que la solicitud salió correctamente.

Por eso utilizamos:

```python
raise HTTPException(...)
```
raise interrumpe la ejecución de esa solicitud y FastAPI devuelve el error HTTP indicado.

### Idea principal
```text
La tarea existe    → return task
La tarea no existe → raise HTTPException(404)
```
El servidor sigue funcionando; solamente termina esa solicitud con un error.


Los casos 16 a 19 completan las operaciones CRUD y luego introducen dependencias.

# 16. Actualizar una tarea con `PUT`
`PUT` actualiza por completo un recurso existente identificado por su URL. A diferencia de `POST`, no crea un ID nuevo: reemplaza la tarea indicada.

El cliente debe enviar todos los datos necesarios; si no existe la tarea, devolvemos `404`.

In [17]:
@app.put("/tasks-memory-update/{task_id}", response_model=TaskResponse)
async def replace_task(task_id: int, task: ValidatedTaskCreated):
    for index, saved_task in enumerate(tasks_memory):
        if saved_task.id == task_id:
            updated_task = TaskResponse(id=task_id, **task.model_dump())
            tasks_memory[index] = updated_task
            return updated_task
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Task not found")

## 16. Ejecucion

Para este caso primero vamos a hacer la actualizacion del `id=1` agregando `title:tarea modificada` y `completed:true`
___
/tasks-memory-update/{task_id}
___
![Actualizar tarea PUT](Images\16%20Actualizar%20tarea%20PUT.png)
___
Luego vamos a revisar en `/task-memory-safe/1` si fue modificado
___
![Actualizar tarea resultado](Images\16%20Actualizar%20tarea%20resultado.png)

## 16. Explicación didáctica

### ¿Para qué se usa `PUT`?

`PUT` se utiliza para reemplazar o actualizar completamente un recurso existente.

Por ejemplo:

`PUT /tasks-memory-update/5`

indica que queremos actualizar la tarea número 5 enviando nuevamente todos sus datos.

### Idea principal

```text
POST → crear un recurso nuevo
PUT  → reemplazar un recurso existente
```

# 17. Modificar solo algunos campos con `PATCH`
`PATCH` actualiza parcialmente un recurso: podemos cambiar solo `completed` sin reenviar el titulo. Por eso los campos de `TaskUpdate` son opcionales.
`exclude_unset = True` conserva únicamente los campos enviados por el cliente; evita que los campos omitidos reemplacen datos existentes por `None`

In [18]:
class TaskUpdate(BaseModel):
    title: str | None = Field(default = None, min_length = 3, max_length=100)
    completed: bool | None = None

@app.patch("/tasks-memory-partial/{task_id}", response_model=TaskResponse)
async def update_task_partially(task_id: int, task:TaskUpdate):
    for index, saved_task in enumerate(tasks_memory):
        if saved_task.id == task_id:
            updated_task = saved_task.model_copy(update=task.model_dump(exclude_unset=True))
            tasks_memory[index] = updated_task
            return updated_task
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Task not found")

## 17. Ejecucion

### ¿Cuándo usar `PUT` y cuándo usar `PATCH`?
Usamos `PUT` cuando el cliente envía la versión completa que debe tener el recurso, reemplazando sus datos actuales.
Usamos `PATCH` cuando solo necesita cambiar uno o algunos campos, por ejemplo marcar una tarea como completada.
En APIs reales, `PATCH` suele ser más práctico para formularios de edición parcial o interruptores; `PUT` para reemplazos explícitos y completos.
___
/tasks-memory-partial/{task_id}
___
![Actualizar tarea parcial docs](Images\17%20Actualizar%20tarea%20parcial%20docs.png)
___
![Actualizar tarea parcial](Images\17%20Actualizar%20tarea%20parcial.png)

## 17. Explicación didáctica

### ¿Para qué se usa `PATCH`?

`PATCH` se utiliza cuando queremos modificar solamente una parte de un recurso.

Por ejemplo, podemos enviar únicamente:

```json
{
    "completed": true
}
```
sin volver a enviar el título.

### `PUT` vs `PATCH`
```text
PUT   → reemplazar completamente
PATCH → modificar parcialmente
```
### Usos típicos

`PATCH` resulta útil para cambios pequeños como:

- marcar una tarea como completada;
- cambiar un número de teléfono;
- modificar una dirección;
- activar o desactivar una opción.

# 18. Eliminar una tarea con `DELETE`
`DELETE` elimina el recurso identificado por la URL. Si se completa, respondemos `204 No Content`: la operacion fue correcta y no necesitamos devolver un cuerpo JSON.

Usamos `pop(index)` para quitar de la lista la tarea encontrada. Si el ID no existe, mantenemos el error `404`.

In [19]:
@app.delete("/tasks-memory-delete/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
async def delete_task(task_id: int):
    for index, saved_task in enumerate(tasks_memory):
        if saved_task.id == task_id:
            tasks_memory.pop(index)
            return
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="task not found")

## 18. Ejecucion

![Eliminar tarea antes](Images\18%20Eliminar%20tarea%20antes.png)
___
/tasks-memory-delete/{task_id}
___
![Eliminar tarea docs](Images\18%20Eliminar%20tarea%20docs.png)
___
![Eliminar tarea despues](Images\18%20Eliminar%20tarea%20despues%201.png)
___
![Eliminar tarea safe despues](Images\18%20Eliminar%20tarea%20despues%202.png)

## 18. Explicación didáctica

### ¿Para qué se usa `DELETE`?

`DELETE` se utiliza para eliminar un recurso.

Por ejemplo:

`DELETE /tasks-memory-delete/5`

significa:

"Eliminá la tarea número 5".

### ¿Qué significa `204 No Content`?

`204 No Content` indica que la operación salió correctamente pero no hay información que devolver.

### Operaciones CRUD

```text
POST   → Create → crear
GET    → Read   → leer
PUT    → Update → reemplazar/actualizar completamente
PATCH  → Update → actualizar parcialmente
DELETE → Delete → eliminar
```
Estas operaciones forman el patrón conocido como CRUD.

# 19. Reutilizar logica con dependencias
Evitar repetir codigo en varios endpoints usando el sistema de dependencias de FastAPI con `Depends()`

`Depends()` ejecuta una funcion antes de la ruta y entrega su resultado. Sirve para centralizar verificaciones compartidas, como permisos, usuarios autenticados o conecciones a una base de datos.

Aqui validamos una clave enviada como parámetro de consulta. Es una demostracion. En un proyecto real no guardariamos una clave sensible asi.

In [20]:
from fastapi import Depends

def verify_api_key(api_key:str):
    if api_key != "learning-key":
        raise HTTPException(status_code=status.HTTP_403_FORBIDDEN, detail="invalid API key")
    return api_key

@app.get("/tasks-protected", response_model=list[TaskResponse])
async def read_protected_tasks(api_key: str = Depends(verify_api_key)):
    return tasks_memory

## 19. Ejecucion

/tasks-protected
___
![Reutilizar logica dependencias](Images\19%20Reutilizar%20logica%20dependencias.png)

## 19. Explicación didáctica

### ¿Para qué sirve `Depends()`?

`Depends()` permite sacar una lógica que se repite de los endpoints y colocarla en una función independiente.

FastAPI ejecuta esa función antes del endpoint y entrega su resultado.

### ¿Qué ocurre en este ejemplo?

Antes de permitir el acceso a `/tasks-protected`, FastAPI ejecuta:

```python
verify_api_key()
```
Si la clave es válida, continúa con el endpoint.

Si no lo es, devuelve un error `403`.

### Usos típicos

`Depends()` se utiliza frecuentemente para:

- autenticación;
- permisos;
- obtener el usuario actual;
- conexiones a bases de datos;
- validaciones compartidas.
### Idea principal
```text
Sin Depends → repetir la misma lógica en muchos endpoints
Con Depends → escribirla una vez y reutilizarla
```

# 20.Recibir informacion mediante cabeceras HTTP
Las cabeceras acompañan una solicitud sin formar parte de la URL ni del cuerpo JSON. Se usan para enviar metadatos como autenticación, tipo de contenido o un identificador de seguimiento.

FastAPI convierte automáticamente `x_request_id` en la cabecera HTTP `X_Request_Id`

In [21]:
from fastapi import Header

@app.get("/tasks-request-info")
async def read_tasks_request_indo(
    user_agent: str | None = Header(default=None),
    x_request_id: str | None = Header(default=None),
):
    return{"user_agent": user_agent, "request_id": x_request_id}

## 20. Ejecucion

___
![Cabeceras HTTP](Images\20%20Cabeceras%20HTTP.png)
___
/tasks-request-info
___
![Cabeceras HTTP docs](Images\20%20Cabeceras%20HTTP%20docs.png)

## 20. Explicacion didactica

### Headers vs parámetros de consulta

Tanto los **headers** como los **parámetros de consulta** permiten enviar información adicional al servidor, pero cumplen funciones distintas.

### Parámetros de consulta

Los parámetros de consulta forman parte de la URL y normalmente indican **qué datos queremos obtener**.

Ejemplo:

```http
GET /tasks?completed=true&limit=10
````

Aquí estamos pidiendo:

* solo tareas completadas;
* como máximo 10 resultados.

Es decir, modifican directamente **qué información devuelve el endpoint**.

### Headers

Los headers no forman parte de la URL. Envían **metadatos sobre la petición**.

Ejemplo:

```http
GET /tasks
Authorization: Bearer abc123
Accept-Language: es
X-Request-Id: req-001
```

En este caso:

* `Authorization` identifica y autentica al usuario;
* `Accept-Language` indica el idioma preferido;
* `X-Request-Id` permite rastrear esa petición en los logs.

Los headers suelen usarse para servicios que afectan **cómo el servidor procesa la petición**, sin cambiar necesariamente el recurso solicitado.

### Headers comunes

| Header            | Uso habitual                                       | Servicio asociado                          |
| ----------------- | -------------------------------------------------- | ------------------------------------------ |
| `Authorization`   | Enviar credenciales o tokens                       | Autenticación y permisos                   |
| `Accept-Language` | Indicar idioma preferido                           | Internacionalización                       |
| `Content-Type`    | Indicar el formato de los datos enviados           | Procesamiento del body                     |
| `Accept`          | Indicar qué formato de respuesta acepta el cliente | Negociación de contenido                   |
| `User-Agent`      | Identificar el programa que realiza la petición    | Compatibilidad, estadísticas o diagnóstico |
| `X-Request-Id`    | Identificar una petición concreta                  | Logs y trazabilidad                        |

### Idea principal

```text
Parámetros de consulta → ¿Qué datos quiero?

Headers               → ¿Con qué contexto debe procesarse la petición?
```

Por ejemplo:

```http
GET /tasks?completed=true
Accept-Language: es
Authorization: Bearer abc123
```

El parámetro:

```text
completed=true
```

indica **qué tareas queremos**.

Los headers indican, por ejemplo:

```text
Authorization   → quién está haciendo la petición
Accept-Language → en qué idioma debería responder el servidor
```



# 21. Permitir solicitudes desde otro origen como CORS
Un origen combina protocolo, dominio y puerto. Por ejemplo `localhost:3000` y `127.0.0.1:8000` son origenes distintos.
CORS permite que un navegador acepte solicitudes de un frontend a nuestra API; no reemplaza autenticacion ni permisos.

In [22]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["*"],
    allow_headers=["*"],
)

## 21. Ejecucion

Servidor web estático temporal de Python. Publica los archivos de la carpeta actual en http://localhost:3000.
No es FastAPI ni ejecuta la API: solo crea el segundo origen necesario para probar CORS desde el navegador.
___
![Servidor Web estatico](Images\21%20Servidor%20web%20estatico.png)
___ 
La consola del navegador muestra que la solicitud `fetch()` fue realizada desde `localhost:3000` y recibió correctamente las cinco tareas desde la API en `127.0.0.1:8000`.
La `Promise` representa la operación asíncrona de espera; una vez resuelta, se muestra el arreglo con la respuesta.
Esto confirma que la configuración CORS permite la comunicación entre ambos orígenes.
___
![Permitir solicitudes CORS](Images\21%20Permitir%20solicitudes%20CORS.png)

## 21. Explicación didáctica

### ¿Qué es un middleware?

Un **middleware** es una capa que se ejecuta entre la llegada de una solicitud y nuestro endpoint.

Puede inspeccionar o modificar la solicitud antes de que llegue a la función y también modificar la respuesta antes de enviarla al cliente.

```text
Cliente
   ↓
Middleware
   ↓
Endpoint
   ↓
Middleware
   ↓
Respuesta
```

Esto permite agregar comportamientos generales sin repetir código en cada endpoint.

### ¿Dónde se suele usar?

Los middlewares suelen utilizarse para tareas que afectan a muchas o todas las rutas de una API:

CORS → controlar qué orígenes pueden acceder desde un navegador.
GZip → comprimir respuestas para reducir la cantidad de datos enviados.
HTTPS Redirect → redirigir automáticamente de HTTP a HTTPS.
Trusted Host → aceptar solicitudes solamente dirigidas a dominios autorizados.
Logs → registrar solicitudes, tiempos de respuesta o errores.
Métricas → medir cantidad de solicitudes y rendimiento.
CORS como middleware

En este caso agregamos:

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["*"],
    allow_headers=["*"],
)

Esto hace que CORSMiddleware revise las solicitudes provenientes del navegador antes de que lleguen a nuestros endpoints.

¿Sirve para comunicación entre servidores?

Un middleware puede procesar solicitudes que provengan de otros servidores, pero CORS específicamente está pensado principalmente para navegadores.

Por ejemplo:

Frontend React
localhost:3000
      ↓
     CORS
      ↓
FastAPI
127.0.0.1:8000

En una comunicación directa:

Servidor A → Servidor B

normalmente CORS no es el mecanismo que controla el acceso, porque CORS es una restricción aplicada por los navegadores.

Idea principal
Middleware → lógica general que se ejecuta alrededor de las solicitudes

CORS            → controlar orígenes en navegadores
GZip            → comprimir respuestas
HTTPSRedirect   → obligar a utilizar HTTPS
TrustedHost     → limitar dominios aceptados
Logs/Métricas   → observar el funcionamiento de la API

# 22. Organizar rutas con `APIRouter`
Cuando una API crece, agrupamos rutas relacionadas en routers: por ejemplo, tareas, usuarios o autenticación. En un proyecto real cada router suele vivir en su propio archivo.
Aquí lo dejamos en una celda para entender la relación: el router registra sus rutas y `app.include_router()` las incorpora a la aplicación principal.

In [23]:
from fastapi import APIRouter


# Router para tareas
tasks_router = APIRouter(
    prefix="/organized-tasks",
    tags=["Tasks"]
)

@tasks_router.get("/summary")
async def read_tasks_summary():
    return {"total_tasks": len(tasks_memory)}


# Router para usuarios
users_router = APIRouter(
    prefix="/organized-users",
    tags=["Users"]
)

@users_router.get("/summary")
async def read_users_summary():
    return {"message": "Resumen de usuarios"}


# Router para autenticación
auth_router = APIRouter(
    prefix="/organized-auth",
    tags=["Authentication"]
)

@auth_router.get("/status")
async def read_auth_status():
    return {"authenticated": True}


# Router para productos
products_router = APIRouter(
    prefix="/organized-products",
    tags=["Products"]
)

@products_router.get("/summary")
async def read_products_summary():
    return {"message": "Resumen de productos"}


# Incorporamos todos los routers a la aplicación principal
app.include_router(tasks_router)
app.include_router(users_router)
app.include_router(auth_router)
app.include_router(products_router)

## 22. Ejecucion

___
![Organizar rutas docs](Images\22%20Organizar%20rutas%20docs.png)

# 22. Explicación didáctica

### ¿Para qué se usa `APIRouter`?

`APIRouter` se usa para **dividir una API grande en grupos de rutas relacionadas**.

Por ejemplo:

```text
tasks_router    → rutas de tareas
users_router    → rutas de usuarios
auth_router     → rutas de autenticación
products_router → rutas de productos
````

Cada router funciona como un **objeto intermedio**: primero registramos allí sus rutas y después lo conectamos a la aplicación principal.

```text
Rutas de tareas
      ↓
tasks_router
      ↓
app.include_router(...)
      ↓
app
```

### ¿Por qué no usar siempre `app` directamente?

En una API pequeña podemos escribir todo así:

```python
@app.get(...)
@app.post(...)
@app.delete(...)
```

Pero cuando hay muchas rutas, `main.py` puede volverse enorme y difícil de mantener.

Con `APIRouter` podemos separar cada grupo.

Por ejemplo:

```text
main.py
routers/
    tasks.py
    users.py
    auth.py
    products.py
```

Cada archivo puede crear su propio router:

```text
tasks.py    → tasks_router
users.py    → users_router
auth.py     → auth_router
products.py → products_router
```

y `main.py` solamente los une a la aplicación principal.

### ¿Qué ocurre al final?

Aunque durante la definición usamos:

```python
@tasks_router.get(...)
@users_router.get(...)
```

al final todos los routers se incorporan a `app`:

```python
app.include_router(tasks_router)
app.include_router(users_router)
app.include_router(auth_router)
app.include_router(products_router)
```

Por eso todas esas rutas terminan formando parte de **la misma API**.

### Parámetros importantes

`prefix` agrega una ruta base a todas las rutas del router.

Por ejemplo:

```python
APIRouter(prefix="/users")
```

junto con:

```python
@users_router.get("/summary")
```

produce:

```text
/users/summary
```

`tags` sirve principalmente para organizar las rutas visualmente dentro de `/docs`.

### ¿Para qué se usa en proyectos reales?

Principalmente para **organización y mantenibilidad**.

Se suele separar por áreas funcionales:

```text
usuarios
productos
pedidos
autenticación
pagos
administración
```

Esto permite que cada parte del backend tenga sus propias rutas y su propio archivo.

### ¿Separa procesos o servidores?

No.

`APIRouter` organiza el código, pero todos los routers siguen ejecutándose normalmente dentro de **la misma aplicación FastAPI y el mismo proceso del servidor**.

```text
Un servidor FastAPI

app
├── tasks_router
├── users_router
├── auth_router
└── products_router
```

Si quisiéramos que usuarios y productos se ejecutaran en **servidores o procesos distintos**, ya estaríamos hablando de otra arquitectura, por ejemplo microservicios.

### Idea principal

```text
APIRouter → separar y organizar grupos de rutas

router → objeto intermedio donde registramos rutas

include_router() → une ese grupo a la aplicación principal

APIRouter NO → crea otro servidor
APIRouter NO → crea otro proceso
```

```

La utilidad real de `APIRouter` es sobre todo esa: **poder convertir un único archivo enorme en módulos pequeños y manejables sin dejar de tener una sola API**.

# 23. Preparar una base de datos con SQLite y SQLAlchemy
La lista en memoria desaparece al reiniciar; SQLite guarda los datos en el archivo local `tasks.db`. SQLAlchemy actúa como ORM: relaciona una clase Python con una tabla de la base de datos.
Esta celda solo prepara la conexión y la tabla. En el siguiente paso crearemos rutas que realmente guarden tareas allí.
Usamos `%pip install sqlalchemy`

In [24]:
from sqlalchemy import Boolean, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker

engine = create_engine("sqlite:///./tasks.db", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine)

class Base(DeclarativeBase):
    pass

class TaskRecord(Base):
    __tablename__ = "tasks"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(100))
    completed: Mapped[bool] = mapped_column(Boolean, default=False)

Base.metadata.create_all(engine)

## 23. Ejecucion

___
![Archivo de la base de datos](Images\23%20Archivo%20base%20datos.png)

## 23. Explicación didáctica

### ¿Qué problema resolvemos?

Hasta ahora guardábamos las tareas en una lista de Python.

El problema es que esa lista vive solamente en memoria:

```text
Servidor funcionando → datos disponibles
Reinicio del servidor → datos perdidos
````

Con SQLite, los datos se guardan en un archivo real:

```text
tasks.db
```

Por eso pueden seguir existiendo aunque reiniciemos la aplicación.

### ¿Qué papel cumple SQLAlchemy?

SQLAlchemy actúa como intermediario entre Python y la base de datos.

En lugar de escribir directamente sentencias SQL todo el tiempo, podemos representar una tabla mediante una clase de Python.

```text
Clase Python  ↔  Tabla SQL
TaskRecord    ↔  tasks
```

A este enfoque se lo conoce como **ORM**.

### ¿Qué objetos importantes se crean?

`engine` representa la conexión con la base de datos:

```python
engine = create_engine("sqlite:///./tasks.db")
```

`SessionLocal` prepara sesiones que luego utilizaremos para leer y escribir datos.

```python
SessionLocal = sessionmaker(bind=engine)
```

`Base` es la clase base que SQLAlchemy utiliza para registrar nuestros modelos.

```python
class Base(DeclarativeBase):
    pass
```

Y `TaskRecord` representa la tabla `tasks`.

### ¿Qué representa `TaskRecord`?

```python
class TaskRecord(Base):
    __tablename__ = "tasks"
```

indica que esta clase corresponde a una tabla llamada:

```text
tasks
```

Sus atributos representan columnas:

```text
id        → identificador
title     → texto de la tarea
completed → estado de la tarea
```

### ¿Qué hace `create_all()`?

```python
Base.metadata.create_all(engine)
```

le indica a SQLAlchemy:

> "Creá en la base de datos las tablas que definimos, si todavía no existen."

Después de ejecutar esta celda aparece:

```text
tasks.db
```

### ¿Por qué todavía no usamos `httpx`?

Porque en este paso solamente preparamos:

```text
conexión
    ↓
sesiones
    ↓
modelo
    ↓
tabla
```

Todavía no existe ningún endpoint que lea o escriba en esa tabla.

Por eso todavía no tiene sentido llamar algo como:

```http
POST /database-tasks
```

### Idea principal

```text
SQLite       → guarda los datos en tasks.db

SQLAlchemy   → conecta Python con la base de datos

TaskRecord   → representa la tabla tasks

SessionLocal → permitirá trabajar con los datos

create_all() → crea las tablas necesarias
```

En el siguiente paso recién conectaremos esta base de datos con FastAPI mediante un endpoint como:

```text
POST /database-tasks
```

y ahí sí tendrá sentido probarlo con `/docs` o `httpx`.


# 24. Crear tareas persistentes en SQLite
La dependencia `get_db()` abre una sesión de base de datos para cada solicitud y la cierra al terminar. La ruta agrega el registro, confirma el cambio con `commit()` y recupera su `id` generado.
Usamos `def` porque SQLAlchemy síncrono realiza operaciones bloqueantes; FastAPI ejecuta estas funciones en un hilo de trabajo.

In [25]:
from collections.abc import Generator
from sqlalchemy.orm import Session

def get_db() -> Generator[Session, None, None]:
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.post("/database-tasks", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
def create_database_task(task: ValidatedTaskCreated, db: Session = Depends(get_db)):
    new_task = TaskRecord(**task.model_dump())
    db.add(new_task)
    db.commit()
    db.refresh(new_task)
    return {"id": new_task.id, "title": new_task.title, "completed": new_task.completed}

## 24. Ejecucion

/database-tasks
___
![Tareas persistentes docs](Images\24%20Tareas%20Persistentes%20docs.png)

## 24. Explicación didáctica

### ¿Qué problema resolvemos?

Hasta ahora ya teníamos la base de datos y la tabla creadas, pero todavía no guardábamos datos desde FastAPI.

En este paso conectamos por primera vez:

```text
Cliente
   ↓
FastAPI
   ↓
SQLAlchemy
   ↓
SQLite
````

### ¿Para qué sirve `get_db()`?

`get_db()` crea una sesión de base de datos para cada solicitud.

Esa sesión es el objeto que usamos para leer o modificar datos.

Al terminar la solicitud:

```python
db.close()
```

cierra la sesión.

Esto evita dejar conexiones abiertas innecesariamente.

### ¿Qué hace `Depends(get_db)`?

```python
db: Session = Depends(get_db)
```

le dice a FastAPI:

> Antes de ejecutar este endpoint, obtené una sesión de base de datos y entregámela en `db`.

### ¿Qué ocurre al crear una tarea?

```text
POST /database-tasks
        ↓
se recibe el JSON
        ↓
se crea TaskRecord
        ↓
db.add()
        ↓
db.commit()
        ↓
SQLite guarda el registro
```

`db.refresh(new_task)` vuelve a leer el registro desde la base para obtener datos generados allí, como el `id`.

### ¿Por qué usamos `def` y no `async def`?

En este ejemplo usamos SQLAlchemy de forma síncrona.

Como esas operaciones pueden bloquear mientras acceden a la base de datos, FastAPI ejecuta la función en un hilo de trabajo para no bloquear el servidor principal.

### Idea principal

```text
get_db()     → abre y cierra la sesión
Depends()    → entrega esa sesión al endpoint
db.add()     → prepara un nuevo registro
db.commit()  → confirma y guarda
db.refresh() → actualiza el objeto con los datos de la base
```

# 25. Consultar tareas persistentes
`select()` construye una consulta SQL mediante SQLAlchemy. La sesión ejecuta esa consulta y convierte los registros obtenidos en objetos `TaskRecord`.
`ConfigDict(from_attributes=True)` permite que Pydantic genere la respuesta leyendo los atributos del objeto de SQLAlchemy.

In [26]:
from pydantic import ConfigDict
from sqlalchemy import select

class DatabaseTaskResponse(BaseModel):
    id: int
    title: str
    completed: bool

    model_config = ConfigDict(from_attributes=True)

@app.get("/database-tasks", response_model=list[DatabaseTaskResponse])
def read_database_tasks(db: Session = Depends(get_db)):
    query = select(TaskRecord).order_by(TaskRecord.id)
    return db.scalars(query).all()

@app.get("/database-tasks/{task_id}", response_model=DatabaseTaskResponse)
def read_database_task(task_id: int, db: Session = Depends(get_db)):
    task = db.get(TaskRecord, task_id)

    if task is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Task not found",
        )

    return task

## 25. Ejecucion

/database-tasks/{task_id}
___
![Consultar persistentes docs](Images\25%20Consultar%20Persistentes%20docs.png)

## 25. Explicación didáctica


### ¿Qué estamos haciendo ahora?

En el paso anterior aprendimos a guardar tareas.

Ahora usamos SQLAlchemy para **leer los registros persistentes** almacenados en SQLite.

Podemos consultar:

```text
todas las tareas
una tarea específica por ID
````

### ¿Qué hace `select()`?

```python
select(TaskRecord)
```

construye una consulta para obtener registros de la tabla representada por `TaskRecord`.

Después:

```python
db.scalars(query).all()
```

ejecuta la consulta y devuelve los registros encontrados como objetos Python.

### Consultar todas las tareas

```http
GET /database-tasks
```

devuelve la colección completa.

Además:

```python
order_by(TaskRecord.id)
```

indica que queremos los resultados ordenados por `id`.

### Consultar una tarea por ID

```http
GET /database-tasks/5
```

utiliza:

```python
db.get(TaskRecord, task_id)
```

para buscar directamente el registro cuya clave primaria coincide con ese ID.

Si no existe, devolvemos:

```text
404 Not Found
```

### ¿Para qué sirve `from_attributes=True`?

SQLAlchemy devuelve objetos `TaskRecord`, pero FastAPI debe convertirlos al modelo de respuesta de Pydantic.

```python
ConfigDict(from_attributes=True)
```

permite que Pydantic lea directamente atributos como:

```text
task.id
task.title
task.completed
```

### Idea principal

```text
select()       → construir una consulta
db.scalars()   → ejecutar y obtener registros
db.get()       → buscar por clave primaria
from_attributes → convertir objetos SQLAlchemy en respuestas Pydantic
```

# 26. Modificar y eliminar registros persistentes
Para actualizar recuperamos el registro, modificamos sus atributos con `setattr()` y confirmamos la transacción mediante `commit()`.
Para eliminar usamos `delete()` y luego `commit()`. En ambos casos respondemos `404` cuando el registro no existe.

In [27]:
@app.patch("/database-tasks/{task_id}", response_model=DatabaseTaskResponse)
def update_database_task(
    task_id: int,
    task_update: TaskUpdate,
    db: Session = Depends(get_db),
):
    saved_task = db.get(TaskRecord, task_id)

    if saved_task is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Task not found",
        )

    update_data = task_update.model_dump(exclude_unset=True)

    for field, value in update_data.items():
        setattr(saved_task, field, value)

    db.commit()
    db.refresh(saved_task)
    return saved_task

@app.delete(
    "/database-tasks/{task_id}",
    status_code=status.HTTP_204_NO_CONTENT,
)
def delete_database_task(task_id: int, db: Session = Depends(get_db)):
    saved_task = db.get(TaskRecord, task_id)

    if saved_task is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Task not found",
        )

    db.delete(saved_task)
    db.commit()

## 26. Ejecucion

/database-tasks/{task_id}
___
![Modificar persistentes docs](Images\26%20Modificar%20Persistentes%20docs.png)
___
![Modificar persistentes](Images\26%20Modificar%20Persistentes.png)
___
/database-tasks/{task_id}
___
![Eliminar persistentes docs](Images\26%20Eliminar%20Persistentes%20docs.png)
___
![Eliminar persistentes Eliminar](Images\26%20Eliminar%20Persistentes.png)

## 26. Explicación didáctica

### ¿Qué operaciones agregamos?

Ahora completamos las operaciones principales sobre los datos persistentes:

```text
PATCH  → modificar
DELETE → eliminar
````

Ya no trabajamos sobre una lista en memoria: los cambios se realizan directamente sobre SQLite.

### ¿Cómo se modifica un registro?

Primero buscamos la tarea:

```python
saved_task = db.get(TaskRecord, task_id)
```

Después obtenemos solamente los campos enviados por el cliente:

```python
task_update.model_dump(exclude_unset=True)
```

y los aplicamos al registro.

### ¿Qué hace `setattr()`?

```python
setattr(saved_task, field, value)
```

permite modificar dinámicamente un atributo.

Por ejemplo, conceptualmente:

```text
field = "completed"
value = True
```

equivale a:

```python
saved_task.completed = True
```

### ¿Cómo se elimina un registro?

```python
db.delete(saved_task)
db.commit()
```

`delete()` marca el objeto para eliminarlo y `commit()` confirma definitivamente el cambio.

### ¿Por qué vuelve a aparecer `commit()`?

Modificar un objeto Python no alcanza.

SQLAlchemy mantiene los cambios dentro de una transacción hasta que hacemos:

```python
db.commit()
```

### Idea principal

```text
db.get()    → buscar el registro
setattr()   → modificar atributos
db.delete() → marcar para eliminar
db.commit() → confirmar el cambio en SQLite
```

# 27. Filtrar, ordenar y paginar consultas
Los parámetros de consulta controlan qué registros obtiene SQLAlchemy. `where()` filtra, `order_by()` ordena y la combinación `offset()`–`limit()` permite paginar.
La validación de FastAPI impide límites negativos o excesivos antes de ejecutar la consulta SQL.

In [28]:
from typing import Literal

@app.get(
    "/database-tasks-search",
    response_model=list[DatabaseTaskResponse],
)
def search_database_tasks(
    completed: bool | None = None,
    search: str | None = None,
    offset: Annotated[int, Query(ge=0)] = 0,
    limit: Annotated[int, Query(ge=1, le=100)] = 10,
    order: Literal["asc", "desc"] = "asc",
    db: Session = Depends(get_db),
):
    query = select(TaskRecord)

    if completed is not None:
        query = query.where(TaskRecord.completed == completed)

    if search:
        query = query.where(TaskRecord.title.ilike(f"%{search}%"))

    if order == "desc":
        query = query.order_by(TaskRecord.id.desc())
    else:
        query = query.order_by(TaskRecord.id.asc())

    query = query.offset(offset).limit(limit)
    return db.scalars(query).all()

## 27. Ejecucion

/database-tasks-search
___
![Filtrar y ordenar persistentes docs](Images\27%20Filtrar%20ordenar%20persistentes%20docs.png)

## 27. Explicación didáctica

### ¿Qué problema resolvemos?

Cuando una tabla tiene muchos registros, normalmente no queremos devolverlos todos.

Podemos permitir que el cliente decida:

```text
qué registros quiere
en qué orden
desde qué posición
cuántos quiere recibir
````

Para eso combinamos parámetros de consulta con SQLAlchemy.

### Filtrar con `where()`

Por ejemplo:

```http
GET /database-tasks-search?completed=true
```

hace que SQLAlchemy agregue una condición:

```python
where(TaskRecord.completed == completed)
```

### Buscar por texto

```text
search=fastapi
```

utiliza:

```python
ilike("%fastapi%")
```

para buscar títulos que contengan ese texto sin distinguir normalmente mayúsculas y minúsculas.

### Ordenar resultados

```text
order=asc
order=desc
```

permite elegir entre:

```python
TaskRecord.id.asc()
TaskRecord.id.desc()
```

### Paginar resultados

```text
offset → desde qué registro empezar
limit  → cuántos registros devolver
```

Por ejemplo:

```http
?offset=20&limit=10
```

significa:

> Saltá los primeros 20 registros y devolveme los siguientes 10.

### Usos típicos

Este patrón se usa mucho en:

* listados de usuarios;
* catálogos de productos;
* pedidos;
* historiales;
* registros de sensores;
* resultados de búsqueda.

### Idea principal

```text
where()    → filtrar
order_by() → ordenar
offset()   → saltar registros
limit()    → limitar resultados
```

# 28. Relaciones entre tablas
Una clave foránea vincula cada tarea con el usuario propietario mediante `user_id`. SQLAlchemy representa esta relación con `relationship()`.
La relación es de uno a muchos: un usuario puede tener varias tareas, pero cada tarea pertenece a un único usuario.

In [29]:
from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

class UserRecord(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(255), unique=True)

    tasks: Mapped[list["UserTaskRecord"]] = relationship(
        back_populates="owner",
        cascade="all, delete-orphan",
    )

class UserTaskRecord(Base):
    __tablename__ = "user_tasks"

    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(100))
    completed: Mapped[bool] = mapped_column(Boolean, default=False)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))

    owner: Mapped["UserRecord"] = relationship(back_populates="tasks")

Base.metadata.create_all(engine)

In [30]:
class UserCreate(BaseModel):
    name: str = Field(min_length=2, max_length=100)
    email: str = Field(min_length=5, max_length=255)

class UserTaskCreate(BaseModel):
    title: str = Field(min_length=3, max_length=100)
    completed: bool = False

class UserTaskResponse(BaseModel):
    id: int
    title: str
    completed: bool
    user_id: int

    model_config = ConfigDict(from_attributes=True)

class UserResponse(BaseModel):
    id: int
    name: str
    email: str
    tasks: list[UserTaskResponse]

    model_config = ConfigDict(from_attributes=True)

In [31]:
@app.post(
    "/database-users",
    response_model=UserResponse,
    status_code=status.HTTP_201_CREATED,
)
def create_database_user(
    user: UserCreate,
    db: Session = Depends(get_db),
):
    new_user = UserRecord(**user.model_dump())
    db.add(new_user)
    db.commit()
    db.refresh(new_user)
    return new_user

@app.post(
    "/database-users/{user_id}/tasks",
    response_model=UserTaskResponse,
    status_code=status.HTTP_201_CREATED,
)
def create_user_task(
    user_id: int,
    task: UserTaskCreate,
    db: Session = Depends(get_db),
):
    user = db.get(UserRecord, user_id)

    if user is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="User not found",
        )

    new_task = UserTaskRecord(user_id=user_id, **task.model_dump())
    db.add(new_task)
    db.commit()
    db.refresh(new_task)
    return new_task

@app.get("/database-users/{user_id}", response_model=UserResponse)
def read_database_user(user_id: int, db: Session = Depends(get_db)):
    user = db.get(UserRecord, user_id)

    if user is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="User not found",
        )

    return user

## 28. Ejecucion

`POST /database-users` crea un usuario nuevo en la tabla `users`.

`name` y `email` los envía el cliente; el `id: 4` lo genera la base de datos al guardar el registro.

`tasks: []` aparece vacío porque este usuario todavía no tiene tareas relacionadas.

La relación se completará cuando creemos una tarea usando ese `id` como `user_id`.

/database-users

![Crear usuario relacionado docs](Images\28%20Crear%20usuario%20relacionado%20docs.png)
___
`POST /database-users/{user_id}/tasks` crea una tarea asociada a un usuario existente.

El `user_id` se envía en la URL; aquí usamos `3`, y por eso en la respuesta también aparece `user_id: 3`.

El `title` y `completed` vienen del cuerpo JSON, mientras que el `id: 9` lo genera la base de datos al guardar la nueva tarea.

Esto refleja la relación: la tarea se guarda en `user_tasks`, pero vinculada al usuario 3 mediante la clave foránea `user_id`.

/database-users/{user_id}/tasks

![Crear tarea asociada a usuario docs](Images\28%20Crear%20tarea%20asociada%20a%20usuario%20docs.png)
___
`GET /database-users/{user_id}` recupera un usuario y sus tareas relacionadas.

Aquí consultamos el usuario `3`; la respuesta devuelve `id: 3` y una lista `tasks`.

Cada tarea mantiene `user_id: 3`, mostrando la relación uno-a-muchos entre ese usuario y sus tareas.

/database-users/{user_id}

![Consultar usuario con tareas docs](Images\28%20Consultar%20usuario%20con%20tareas%20docs.png)

## 28. Explicación didáctica

### ¿Qué problema resolvemos?

Hasta ahora trabajábamos con tablas independientes.

En aplicaciones reales, los datos suelen estar relacionados.

Por ejemplo:

```text
Usuario
  ↓
Tareas
````

Un usuario puede tener varias tareas y cada tarea pertenece a un único usuario.

### ¿Qué es una clave foránea?

En `UserTaskRecord` aparece:

```python
user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
```

Esto indica que `user_id` debe apuntar al `id` de un usuario existente en la tabla `users`.

Conceptualmente:

```text
users
id = 3
name = Ana

user_tasks
id = 10
title = Estudiar
user_id = 3
```

La tarea 10 pertenece al usuario 3.

### ¿Qué hace `relationship()`?

`ForeignKey` crea la relación a nivel de base de datos.

`relationship()` permite navegar esa relación desde Python.

Por ejemplo:

```text
user.tasks
```

devuelve las tareas del usuario.

Y:

```text
task.owner
```

permite acceder al usuario propietario de una tarea.

### ¿Qué relación estamos creando?

Es una relación de **uno a muchos**:

```text
1 usuario
   ↓
muchas tareas
```

### ¿Para qué sirve `cascade="all, delete-orphan"`?

Indica que las tareas dependen del usuario.

Si una tarea queda desvinculada de su usuario o el usuario se elimina según esa relación, SQLAlchemy puede eliminar también esos registros relacionados.

### ¿Cómo se usa en la API?

Podemos crear una tarea directamente asociada a un usuario:

```http
POST /database-users/5/tasks
```

Eso significa:

> Crear una tarea cuyo propietario es el usuario 5.

### Usos típicos

Este tipo de relación aparece constantemente:

```text
usuario → pedidos
cliente → facturas
autor → publicaciones
proyecto → tareas
empresa → empleados
```

### Idea principal

```text
ForeignKey()   → relación en la base de datos
relationship() → relación accesible desde Python

UserRecord.tasks → tareas del usuario
UserTaskRecord.owner → propietario de la tarea
```

# 29. Restricciones y transacciones
La restricción `unique=True` impide guardar dos usuarios con el mismo correo. Si `commit()` falla, debemos ejecutar `rollback()` para devolver la sesión a un estado utilizable.
Respondemos `409 Conflict` porque la solicitud es válida, pero entra en conflicto con un registro existente.

In [32]:
from sqlalchemy.exc import IntegrityError

@app.post(
    "/database-users-safe",
    response_model=UserResponse,
    status_code=status.HTTP_201_CREATED,
)
def create_database_user_safe(
    user: UserCreate,
    db: Session = Depends(get_db),
):
    new_user = UserRecord(**user.model_dump())
    db.add(new_user)

    try:
        db.commit()
    except IntegrityError:
        db.rollback()
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="Email already registered",
        )

    db.refresh(new_user)
    return new_user

## 29. Ejecucion

Al intentar registrar nuevamente un email que ya existe, la base de datos rechaza la operación.

El endpoint captura ese conflicto y responde con `409 Conflict`.

El mensaje `Email already registered` informa al cliente cuál fue el problema.

/database-users-safe
___
![Email duplicado 409 docs](Images\29%20Email%20duplicado%20409%20docs.png)

## 29. Explicación didáctica

### ¿Qué problema resolvemos?

Las bases de datos pueden imponer reglas para proteger la consistencia de los datos.

En este caso:

```python
email: Mapped[str] = mapped_column(String(255), unique=True)
````

indica que no puede haber dos usuarios con el mismo correo.

### ¿Qué ocurre si intentamos repetir un email?

La base de datos rechaza el `commit()` y SQLAlchemy genera:

```python
IntegrityError
```

Por eso usamos:

```python
try:
    db.commit()
except IntegrityError:
```

### ¿Para qué sirve `rollback()`?

Cuando una transacción falla, la sesión queda en un estado de error.

```python
db.rollback()
```

cancela los cambios pendientes y devuelve la sesión a un estado utilizable.

Conceptualmente:

```text
Intentamos guardar
      ↓
¿La operación es válida?
   ↓ sí          ↓ no
commit()      rollback()
```

### ¿Qué es una transacción?

Una transacción agrupa cambios que deben completarse correctamente antes de considerarse definitivos.

`commit()` confirma los cambios.

`rollback()` los deshace si algo falla.

### ¿Por qué devolvemos `409 Conflict`?

La solicitud está bien formada, pero entra en conflicto con un dato existente.

Por ejemplo:

```text
Email ya registrado
```

Por eso usamos:

```text
409 Conflict
```

en lugar de un error de validación del JSON.

### Usos típicos de restricciones

Las bases de datos suelen impedir:

* emails duplicados;
* nombres de usuario repetidos;
* claves foráneas inexistentes;
* valores nulos donde no están permitidos;
* combinaciones duplicadas de ciertos campos.

### Idea principal

```text
unique=True    → impide duplicados
commit()       → confirma la transacción
IntegrityError → detecta una violación de integridad
rollback()     → revierte la transacción fallida
409 Conflict   → informa el conflicto al cliente
```

___
___
# Ejecutamos

In [33]:
import threading, uvicorn
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000), daemon=True).start()

# Para pruebas agregamos la siguiente libreria y URL base.

In [34]:
import httpx

base_url = "http://127.0.0.1:8000"

INFO:     Started server process [22904]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


### Caso 8 - Crear una tarea simple

In [35]:
response = httpx.post(
    f"{base_url}/tasks-created",
    json={"title": "Estudiar POST", "completed": False},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:59411 - "POST /tasks-created HTTP/1.1" 200 OK
200 {'message': 'Tarea creada', 'task': {'title': 'Estudiar POST', 'completed': False}}


### Caso 9 - Probar validación

In [36]:
valid_response = httpx.post(f"{base_url}/tasks-validated", json={"title": "Tarea válida"})
invalid_response = httpx.post(f"{base_url}/tasks-validated", json={"title": "A"})

print(valid_response.status_code, valid_response.json())
print(invalid_response.status_code, invalid_response.json())

INFO:     127.0.0.1:59413 - "POST /tasks-validated HTTP/1.1" 200 OK
INFO:     127.0.0.1:59414 - "POST /tasks-validated HTTP/1.1" 422 Unprocessable Entity
200 {'message': 'Tarea validada', 'task': {'title': 'Tarea válida', 'completed': False}}
422 {'detail': [{'type': 'string_too_short', 'loc': ['body', 'title'], 'msg': 'String should have at least 3 characters', 'input': 'A', 'ctx': {'min_length': 3}}]}


Caso 10 - Comprobar 201 Created

In [37]:
response = httpx.post(
    f"{base_url}/tasks-created-with-status",
    json={"title": "Comprobar código 201"},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:59415 - "POST /tasks-created-with-status HTTP/1.1" 201 Created
201 {'message': 'Tarea creada correctamente', 'task': {'title': 'Comprobar código 201', 'completed': False}}


### Caso 11 - Comprobar el modelo de respuesta

In [38]:
response = httpx.post(
    f"{base_url}/tasks-response-model",
    json={"title": "Comprobar respuesta estructurada", "completed": True},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:59416 - "POST /tasks-response-model HTTP/1.1" 201 Created
201 {'id': 1, 'title': 'Comprobar respuesta estructurada', 'completed': True}


### Caso 12 - Cargar seis tareas en memoria

In [39]:
titles = ["Leer documentación", "Crear rutas", "Probar POST", "Usar PUT", "Usar PATCH", "Documentar API"]

for title in titles:
    response = httpx.post(f"{base_url}/tasks-memory", json={"title": title})
    print(response.status_code, response.json())

INFO:     127.0.0.1:59418 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 1, 'title': 'Leer documentación', 'completed': False}
INFO:     127.0.0.1:59419 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 2, 'title': 'Crear rutas', 'completed': False}
INFO:     127.0.0.1:59420 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 3, 'title': 'Probar POST', 'completed': False}
INFO:     127.0.0.1:59421 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 4, 'title': 'Usar PUT', 'completed': False}
INFO:     127.0.0.1:59426 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 5, 'title': 'Usar PATCH', 'completed': False}
INFO:     127.0.0.1:59427 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 6, 'title': 'Documentar API', 'completed': False}


### Caso 13 - Listar las tareas creadas

In [40]:
response = httpx.get(f"{base_url}/tasks-memory")
print(response.status_code)
print(response.json())

INFO:     127.0.0.1:59429 - "GET /tasks-memory HTTP/1.1" 200 OK
200
[{'id': 1, 'title': 'Leer documentación', 'completed': False}, {'id': 2, 'title': 'Crear rutas', 'completed': False}, {'id': 3, 'title': 'Probar POST', 'completed': False}, {'id': 4, 'title': 'Usar PUT', 'completed': False}, {'id': 5, 'title': 'Usar PATCH', 'completed': False}, {'id': 6, 'title': 'Documentar API', 'completed': False}]


### Caso 14 - Buscar una tarea por ID

In [41]:
tasks = httpx.get(f"{base_url}/tasks-memory").json()
task_id = tasks[0]["id"]

response = httpx.get(f"{base_url}/tasks-memory/{task_id}")
print(response.status_code, response.json())

INFO:     127.0.0.1:59430 - "GET /tasks-memory HTTP/1.1" 200 OK
INFO:     127.0.0.1:59435 - "GET /tasks-memory/1 HTTP/1.1" 200 OK
200 {'id': 1, 'title': 'Leer documentación', 'completed': False}


### Caso 15 - Solicitar una tarea inexistente

In [42]:
response = httpx.get(f"{base_url}/tasks-memory-safe/-1")
print(response.status_code, response.json())

INFO:     127.0.0.1:59437 - "GET /tasks-memory-safe/-1 HTTP/1.1" 404 Not Found
404 {'detail': 'Task not found'}


### Caso 17 - Modificar parcialmente una tarea con PATCH

In [43]:
tasks = httpx.get(f"{base_url}/tasks-memory").json()
task_id = tasks[5]["id"]

response = httpx.patch(
    f"{base_url}/tasks-memory-partial/{task_id}",
    json={"completed": True},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:59438 - "GET /tasks-memory HTTP/1.1" 200 OK
INFO:     127.0.0.1:59440 - "PATCH /tasks-memory-partial/6 HTTP/1.1" 200 OK
200 {'id': 6, 'title': 'Documentar API', 'completed': True}


### Caso 18 - Eliminar una tarea con `DELETE`

In [44]:
tasks = httpx.get(f"{base_url}/tasks-memory").json()
task_id = tasks[-1]["id"]

response = httpx.delete(f"{base_url}/tasks-memory-delete/{task_id}")
print(response.status_code)

print(httpx.get(f"{base_url}/tasks-memory").json())

INFO:     127.0.0.1:59441 - "GET /tasks-memory HTTP/1.1" 200 OK
INFO:     127.0.0.1:59442 - "DELETE /tasks-memory-delete/6 HTTP/1.1" 204 No Content
204
INFO:     127.0.0.1:59443 - "GET /tasks-memory HTTP/1.1" 200 OK
[{'id': 1, 'title': 'Leer documentación', 'completed': False}, {'id': 2, 'title': 'Crear rutas', 'completed': False}, {'id': 3, 'title': 'Probar POST', 'completed': False}, {'id': 4, 'title': 'Usar PUT', 'completed': False}, {'id': 5, 'title': 'Usar PATCH', 'completed': False}]


### Caso 19 - Reutilizar lógica con dependencias.

In [45]:
response = httpx.get(f"{base_url}/tasks-protected", params = {"api_key": "learning-key"})
print(response.status_code, response.json())

response = httpx.get(f"{base_url}/tasks-protected", params = {"api_key": "wrong-key"})
print(response.status_code, response.json())

INFO:     127.0.0.1:59445 - "GET /tasks-protected?api_key=learning-key HTTP/1.1" 200 OK
200 [{'id': 1, 'title': 'Leer documentación', 'completed': False}, {'id': 2, 'title': 'Crear rutas', 'completed': False}, {'id': 3, 'title': 'Probar POST', 'completed': False}, {'id': 4, 'title': 'Usar PUT', 'completed': False}, {'id': 5, 'title': 'Usar PATCH', 'completed': False}]
INFO:     127.0.0.1:59446 - "GET /tasks-protected?api_key=wrong-key HTTP/1.1" 403 Forbidden
403 {'detail': 'invalid API key'}


### Caso 20 - Recibir información mediante cabeceras HTTP

In [46]:
response = httpx.get(
    f"{base_url}/tasks-request-info",
    headers = {"X-Request-Id": "notebook-001"},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:59447 - "GET /tasks-request-info HTTP/1.1" 200 OK
200 {'user_agent': 'python-httpx/0.28.1', 'request_id': 'notebook-001'}


### Caso 21 - Permitir solicitudes desde otro origen como CORS

In [47]:
response = httpx.options(f"{base_url}/tasks-memory", headers={
    "Origin": "http://localhost:3000",
    "Access-Control-Request-Method": "POST",
})
print(response.status_code, response.headers.get("access-control-allow-origin"))

INFO:     127.0.0.1:59448 - "OPTIONS /tasks-memory HTTP/1.1" 200 OK
200 http://localhost:3000


### Caso 22 - Organizar rutas con `APIRouter`

In [48]:
response = httpx.get(f"{base_url}/organized-tasks/summary")
print(response.status_code, response.json())

INFO:     127.0.0.1:59453 - "GET /organized-tasks/summary HTTP/1.1" 200 OK
200 {'total_tasks': 5}


### Caso 24 - Crear tareas persistentes en SQLite

In [49]:
response = httpx.post(f"{base_url}/database-tasks", json={"title": "Persistir en SQLite"})
print(response.status_code, response.json())

INFO:     127.0.0.1:59455 - "POST /database-tasks HTTP/1.1" 201 Created
201 {'id': 14, 'title': 'Persistir en SQLite', 'completed': False}


### Caso 25 - Consultar tareas persistentes

In [50]:
response = httpx.get(f"{base_url}/database-tasks")
tasks = response.json()

print(response.status_code, tasks)

if tasks:
    task_id = tasks[0]["id"]
    response = httpx.get(f"{base_url}/database-tasks/{task_id}")
    print(response.status_code, response.json())

INFO:     127.0.0.1:59456 - "GET /database-tasks HTTP/1.1" 200 OK
200 [{'id': 2, 'title': 'string', 'completed': False}, {'id': 3, 'title': 'string', 'completed': False}, {'id': 4, 'title': 'string', 'completed': False}, {'id': 5, 'title': 'Modificado', 'completed': True}, {'id': 6, 'title': 'string', 'completed': False}, {'id': 7, 'title': 'string', 'completed': False}, {'id': 8, 'title': 'string', 'completed': False}, {'id': 9, 'title': 'string', 'completed': False}, {'id': 10, 'title': 'string', 'completed': False}, {'id': 11, 'title': 'string', 'completed': False}, {'id': 12, 'title': 'Persistir en SQLite', 'completed': False}, {'id': 13, 'title': 'Estudiar consultas SQL', 'completed': True}, {'id': 14, 'title': 'Persistir en SQLite', 'completed': False}]
INFO:     127.0.0.1:59458 - "GET /database-tasks/2 HTTP/1.1" 200 OK
200 {'id': 2, 'title': 'string', 'completed': False}


### Caso 26 - Modificar y eliminar registros persistentes

In [51]:
created = httpx.post(
    f"{base_url}/database-tasks",
    json={"title": "Tarea temporal para modificar"},
).json()

task_id = created["id"]

updated = httpx.patch(
    f"{base_url}/database-tasks/{task_id}",
    json={"completed": True},
)
print(updated.status_code, updated.json())

deleted = httpx.delete(f"{base_url}/database-tasks/{task_id}")
print(deleted.status_code)

INFO:     127.0.0.1:59459 - "POST /database-tasks HTTP/1.1" 201 Created
INFO:     127.0.0.1:59461 - "PATCH /database-tasks/15 HTTP/1.1" 200 OK
200 {'id': 15, 'title': 'Tarea temporal para modificar', 'completed': True}
INFO:     127.0.0.1:59463 - "DELETE /database-tasks/15 HTTP/1.1" 204 No Content
204


### Caso 27 - Filtrar, ordenar y paginar consultas

In [52]:
httpx.post(
    f"{base_url}/database-tasks",
    json={"title": "Estudiar consultas SQL", "completed": True},
)

response = httpx.get(
    f"{base_url}/database-tasks-search",
    params={
        "completed": True,
        "search": "SQL",
        "offset": 0,
        "limit": 5,
        "order": "desc",
    },
)

print(response.status_code, response.json())

INFO:     127.0.0.1:59464 - "POST /database-tasks HTTP/1.1" 201 Created
INFO:     127.0.0.1:59465 - "GET /database-tasks-search?completed=true&search=SQL&offset=0&limit=5&order=desc HTTP/1.1" 200 OK
200 [{'id': 15, 'title': 'Estudiar consultas SQL', 'completed': True}, {'id': 13, 'title': 'Estudiar consultas SQL', 'completed': True}]


### Caso 28 - Relaciones entre tablas

In [53]:
from uuid import uuid4

email = f"student-{uuid4()}@example.com"

user_response = httpx.post(
    f"{base_url}/database-users",
    json={"name": "Ramiro", "email": email},
)

user = user_response.json()
user_id = user["id"]
print(user_response.status_code, user)

task_response = httpx.post(
    f"{base_url}/database-users/{user_id}/tasks",
    json={"title": "Relacionar tablas"},
)
print(task_response.status_code, task_response.json())

response = httpx.get(f"{base_url}/database-users/{user_id}")
print(response.status_code, response.json())

INFO:     127.0.0.1:59467 - "POST /database-users HTTP/1.1" 201 Created
201 {'id': 8, 'name': 'Ramiro', 'email': 'student-e36fd119-4f19-42eb-bbeb-b6151b615c7c@example.com', 'tasks': []}
INFO:     127.0.0.1:59471 - "POST /database-users/8/tasks HTTP/1.1" 201 Created
201 {'id': 10, 'title': 'Relacionar tablas', 'completed': False, 'user_id': 8}
INFO:     127.0.0.1:59473 - "GET /database-users/8 HTTP/1.1" 200 OK
200 {'id': 8, 'name': 'Ramiro', 'email': 'student-e36fd119-4f19-42eb-bbeb-b6151b615c7c@example.com', 'tasks': [{'id': 10, 'title': 'Relacionar tablas', 'completed': False, 'user_id': 8}]}


### Caso 29 - Restricciones y transacciones

In [54]:
from uuid import uuid4

repeated_user = {
    "name": "Repeated user",
    "email": f"repeated-{uuid4()}@example.com",
}

first_response = httpx.post(
    f"{base_url}/database-users-safe",
    json=repeated_user,
)

second_response = httpx.post(
    f"{base_url}/database-users-safe",
    json=repeated_user,
)

print(first_response.status_code, first_response.json())
print(second_response.status_code, second_response.json())

INFO:     127.0.0.1:59475 - "POST /database-users-safe HTTP/1.1" 201 Created
INFO:     127.0.0.1:59476 - "POST /database-users-safe HTTP/1.1" 409 Conflict
201 {'id': 9, 'name': 'Repeated user', 'email': 'repeated-0a6eee91-fbdc-4a20-8d8c-78a4efc21690@example.com', 'tasks': []}
409 {'detail': 'Email already registered'}
